# 예제 franka_ex05: FR3 Cartesian 경로 — 직선 보간 (self-contained)

Franka FR3(7-DOF) 의 끝단(`fr3_hand_tcp`) 을 **공간상의 직선** 으로 이동시키는 예제.

## 이 노트북이 강조하는 것 — "계획 ≠ 실행" 분리 모델

ex03(joint goal) · ex04(pose goal) 는 둘 다 `MoveGroup` 액션 하나로 **계획+실행**을
한 번에 처리했고, 결과는 SUCCESS/FAIL 이진이었다. ex05 는 그 패러다임이 둘로 쪼개진다.

| 단계 | 무엇이 | 결과 |
|---|---|---|
| **계획** | `compute_cartesian_path` 서비스 | trajectory 객체 + `fraction` (달성률) |
| **게이트** | `fraction >= 0.8` 검사 | 통과 / 스킵 |
| **실행** | `ExecuteTrajectory` 액션 | 실제 컨트롤러로 전송 |

`MoveGroup` 액션의 RRT/PRM 경로는 직선이 아닐 수 있지만, `compute_cartesian_path` 는
**공간상 직선** trajectory 를 보장한다. 그리고 SUCCESS/FAIL 이 아니라 "얼마나 풀렸는가"
가 비율로 나오므로, 사용자가 안전 임계값으로 게이트를 걸 수 있다 — 이게 본 예제의 핵심.

## 이전 예제(ex03/ex04)와 다른 점
- 끝단 링크: `fr3_hand_tcp`, planning group: `fr3_arm`
- `home`(올-제로) 자세가 SRDF 에 없다 → 시작/복귀는 `ready`
- Gazebo Sim 환경이므로 `use_sim_time=True`
- 7-DOF redundancy 덕에 같은 직선 경로도 IK 해가 여러 개 → 특이점 회피 잘 됨

## 노트북 구성

1. **로봇 상수**
2. **핵심 — 계획 / 실행 / 게이트 통합 함수** ← 이 노트북의 본질
3. **핵심을 쓰기 위한 설정** — ROS init, 노드, 클라이언트, SRDF, Pose / MoveGroup / 마커 헬퍼
4. **실행 시나리오** — `ready` → 시작점 → 정사각형 → 직선 하강 → 복귀

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는다.

> ⚠ 다른 로봇용 MoveIt launch 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

RViz 가 뜨면 **`MarkerArray` Display 를 추가하고 Topic 을 `/cartesian_path_markers` 로 설정**한다.
Fixed Frame 은 `fr3_link0` 로 둔다.

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab franka_ex05_cartesian_path.ipynb
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

## 1. 로봇 상수

핵심 함수와 설정 양쪽이 모두 참조하므로 가장 먼저 정의한다. `franka_description/robots/fr3/fr3.srdf.xacro` 의 SRDF 값과 일치한다.

In [ ]:
PLANNING_GROUP    = 'fr3_arm'
REFERENCE_FRAME   = 'fr3_link0'
END_EFFECTOR_LINK = 'fr3_hand_tcp'
ARM_JOINTS        = ['fr3_joint1', 'fr3_joint2', 'fr3_joint3',
                     'fr3_joint4', 'fr3_joint5', 'fr3_joint6',
                     'fr3_joint7']
MARKER_TOPIC      = '/cartesian_path_markers'

## 2. 핵심 — "계획 ≠ 실행" 분리 모델

이 노트북에서 가장 먼저 정의해야 하는 함수들. 셋이 한 묶음으로 본 예제의 본질을 이룬다.

- `compute_cartesian_path()` — 직선 보간 **계획만** (서비스)
- `execute_trajectory()` — 계획된 trajectory 의 **실행만** (액션)
- `plan_and_run_cartesian()` — 위 둘을 `fraction` 게이트로 묶은 통합 함수

함수들은 뒤 셀에서 만드는 `node` / `cart_client` / `execute_client` / `joint_state` 를 참조한다.
함수 *정의* 시점엔 lookup 하지 않으므로 객체가 아직 없어도 OK — 호출은 4 절(시나리오) 에서 일어난다.

### 2-1. 핵심에 필요한 import

In [ ]:
import rclpy
from moveit_msgs.srv import GetCartesianPath
from moveit_msgs.action import ExecuteTrajectory
from moveit_msgs.msg import RobotState, MoveItErrorCodes

### 2-2. `compute_cartesian_path()` — 직선 보간 계획만

| 입력 | 의미 |
|---|---|
| `waypoints` | 끝단이 거쳐야 할 `Pose` 들 — 사이는 직선으로 채운다 |
| `max_step` | 직선 보간 간격 (m) — 작을수록 부드럽고 IK 호출 횟수↑ |
| `avoid_collisions` | True 면 IK 단계에서 충돌 검사 |
| `start_state` | 보통 현재 `joint_states` (없으면 SRDF 기본값) |

응답은 `(trajectory, fraction)`. `fraction == 1.0` 이면 모든 waypoint 까지 IK 풀었다는 뜻.
중간에 특이점/IK 실패 가 있으면 0.x 가 나오므로 임계값(예: 0.8) 으로 필터링한다.

**중요**: 이 함수가 끝나도 로봇은 아직 안 움직인다. 결과물은 trajectory 객체뿐.

In [ ]:
def get_current_robot_state() -> RobotState:
    rs = RobotState()
    if joint_state['msg'] is not None:
        rs.joint_state = joint_state['msg']
    return rs

def compute_cartesian_path(waypoints, max_step: float = 0.01,
                           avoid_collisions: bool = True,
                           vel: float = 0.2, acc: float = 0.2):
    request = GetCartesianPath.Request()
    request.header.frame_id = REFERENCE_FRAME
    request.group_name = PLANNING_GROUP
    request.link_name = END_EFFECTOR_LINK
    request.waypoints = list(waypoints)
    request.max_step = max_step
    request.avoid_collisions = avoid_collisions
    request.max_velocity_scaling_factor = vel
    request.max_acceleration_scaling_factor = acc
    request.start_state = get_current_robot_state()

    future = cart_client.call_async(request)
    rclpy.spin_until_future_complete(node, future)
    response = future.result()

    if response.error_code.val == MoveItErrorCodes.SUCCESS:
        node.get_logger().info(
            f'Cartesian 계획 성공 (달성률: {response.fraction*100:.1f}%)'
        )
        return response.solution, response.fraction
    node.get_logger().error(
        f'Cartesian 계획 실패 error_code={response.error_code.val}'
    )
    return None, 0.0

### 2-3. `execute_trajectory()` — 계획된 trajectory 만 실행

`MoveGroup` 액션과 달리 Cartesian 서비스는 **계획만** 한다.
결과 trajectory 를 실제로 컨트롤러에 보내려면 `ExecuteTrajectory` 액션이 필요하다.

이 분리 덕분에 — 계획과 실행 사이에 fraction 검사, 사용자 확인, 미리보기 등을
끼워 넣을 수 있다.

In [ ]:
def execute_trajectory(trajectory) -> bool:
    goal = ExecuteTrajectory.Goal()
    goal.trajectory = trajectory

    send_future = execute_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, send_future)
    handle = send_future.result()
    if handle is None or not handle.accepted:
        node.get_logger().error('ExecuteTrajectory 목표 거부됨')
        return False

    result_future = handle.get_result_async()
    rclpy.spin_until_future_complete(node, result_future)
    code_val = result_future.result().result.error_code.val
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'궤적 실행 실패 error_code={code_val}')
    return ok

### 2-4. `plan_and_run_cartesian()` — fraction 게이트로 묶기

`compute_cartesian_path` → fraction 검사 → `execute_trajectory` 의 흐름을 한 함수로.
시나리오(4 절) 에서는 이 함수 한 줄이면 끝나도록 설계했다.

`fraction < 1.0` 은 IK 실패 / 특이점 등으로 일부 waypoint 까지만 풀렸다는 뜻.
이진 SUCCESS/FAIL 이 아니라 **"얼마나 풀렸는가"** 를 보고 판단할 수 있다는 게
`MoveGroup` 액션 (ex03 / ex04) 과 가장 다른 점.

In [ ]:
def plan_and_run_cartesian(waypoints, label: str = 'Path',
                           max_step: float = 0.02,
                           min_fraction: float = 0.8) -> bool:
    trajectory, fraction = compute_cartesian_path(waypoints, max_step=max_step)
    node.get_logger().info(f'[{label}] 달성률: {fraction*100:.1f}%')

    if trajectory is None or fraction < min_fraction:
        node.get_logger().warn(
            f'[{label}] 달성률 미달 ({fraction*100:.1f}% < {min_fraction*100:.0f}%) — 실행 스킵'
        )
        return False
    return execute_trajectory(trajectory)

## 3. 핵심을 쓰기 위한 설정

여기부터는 위 핵심 함수들이 참조하는 객체와 보조 헬퍼를 만든다.

- ROS 2 초기화 / 노드 / 액션·서비스 클라이언트 / `joint_states` 구독
- 서버 준비 대기
- SRDF 에서 `ready` 자세 읽기
- `Pose` 헬퍼 (Euler ↔ Quaternion)
- `MoveGroup` 액션 헬퍼 — `ready` 와 정사각형 시작점으로 옮기기 위함
- RViz 마커 헬퍼 — 경로 미리보기와 결과 색상

### 3-1. ROS 2 초기화 + 노드 + 클라이언트

In [ ]:
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from moveit_msgs.action import MoveGroup
from visualization_msgs.msg import MarkerArray

try:
    rclpy.init()
except RuntimeError:
    pass

node = Node(
    'franka_ex05_cartesian_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
move_client    = ActionClient(node, MoveGroup, 'move_action')
execute_client = ActionClient(node, ExecuteTrajectory, 'execute_trajectory')
cart_client    = node.create_client(GetCartesianPath, 'compute_cartesian_path')
marker_pub     = node.create_publisher(MarkerArray, MARKER_TOPIC, 10)

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== franka_ex05 노트북 노드 생성 완료 ===')

### 3-2. 액션 / 서비스 / `/joint_states` 준비 대기

In [ ]:
import time

def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    if not execute_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('ExecuteTrajectory 액션 서버 연결 실패')
    if not cart_client.wait_for_service(timeout_sec=timeout_sec):
        raise RuntimeError('compute_cartesian_path 서비스 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info('move/execute action + cartesian svc + /joint_states 준비됨')

wait_for_ready()

### 3-3. SRDF 에서 `ready` 자세 읽어오기

FR3 SRDF 에는 `home` 이 없고 `ready` / `extended` 만 있다.

In [ ]:
from rclpy.parameter_client import AsyncParameterClient
import xml.etree.ElementTree as ET

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

def load_named_pose(name: str, timeout_sec: float = 10.0) -> dict:
    return parse_named_pose(fetch_srdf_xml(timeout_sec), name, PLANNING_GROUP)

ready_target = load_named_pose('ready')
node.get_logger().info(f'ready: {ready_target}')

### 3-4. Pose 헬퍼 — Euler ↔ Quaternion

In [ ]:
import math
import tf_transformations
from geometry_msgs.msg import Pose, Point, Quaternion

def euler_to_quaternion(roll: float, pitch: float, yaw: float) -> Quaternion:
    q = tf_transformations.quaternion_from_euler(roll, pitch, yaw)
    return Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])

def make_pose(x: float, y: float, z: float,
              roll: float = 0.0, pitch: float = 0.0, yaw: float = 0.0) -> Pose:
    pose = Pose()
    pose.position = Point(x=x, y=y, z=z)
    pose.orientation = euler_to_quaternion(roll, pitch, yaw)
    return pose

### 3-5. `MoveGroup` 액션 헬퍼 — 시작점 이동용

정사각형 계획 전에 `ready`(조인트 목표) 와 시작점(자세 목표) 으로 옮긴다.

In [ ]:
from moveit_msgs.msg import (
    Constraints, JointConstraint,
    PositionConstraint, OrientationConstraint, BoundingVolume,
    MotionPlanRequest, PlanningOptions,
)
from shape_msgs.msg import SolidPrimitive
from geometry_msgs.msg import Vector3

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    c = Constraints()
    for jname, val in joint_values.items():
        c.joint_constraints.append(JointConstraint(
            joint_name=jname, position=val,
            tolerance_above=tol, tolerance_below=tol, weight=1.0,
        ))
    return c

def make_position_constraint(pose: Pose, tol: float = 0.01) -> PositionConstraint:
    pc = PositionConstraint()
    pc.header.frame_id = REFERENCE_FRAME
    pc.link_name = END_EFFECTOR_LINK
    pc.target_point_offset = Vector3(x=0.0, y=0.0, z=0.0)
    bv = BoundingVolume()
    sphere = SolidPrimitive()
    sphere.type = SolidPrimitive.SPHERE
    sphere.dimensions = [tol]
    bv.primitives.append(sphere)
    sp = Pose()
    sp.position = Point(x=pose.position.x, y=pose.position.y, z=pose.position.z)
    sp.orientation.w = 1.0
    bv.primitive_poses.append(sp)
    pc.constraint_region = bv
    pc.weight = 1.0
    return pc

def make_orientation_constraint(pose: Pose, tol: float = 0.01) -> OrientationConstraint:
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    oc.orientation = pose.orientation
    oc.absolute_x_axis_tolerance = tol
    oc.absolute_y_axis_tolerance = tol
    oc.absolute_z_axis_tolerance = tol
    oc.weight = 1.0
    return oc

def make_plan_request(vel: float, acc: float,
                      attempts: int = 5, plan_time: float = 10.0) -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.num_planning_attempts = attempts
    req.allowed_planning_time = plan_time
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    return req

def send_move_goal_and_wait(req: MotionPlanRequest) -> int:
    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=False, replan=True, replan_attempts=3)
    send_future = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, send_future)
    handle = send_future.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED
    result_future = handle.get_result_async()
    rclpy.spin_until_future_complete(node, result_future)
    return result_future.result().result.error_code.val

def go_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val = send_move_goal_and_wait(req)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'joint goal 실패 error_code={code_val}')
    return ok

def go_to_pose_goal(pose: Pose, vel: float = 0.2, acc: float = 0.2) -> bool:
    req = make_plan_request(vel, acc)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val = send_move_goal_and_wait(req)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'pose goal 실패 error_code={code_val} (IK 해 없음 가능)')
    return ok

### 3-6. RViz 마커 헬퍼 — 경로 미리보기 + 결과 표시

| 색상 | 의미 |
|---|---|
| 시안 (`COLOR_SQUARE`) | 정사각형 미리보기 |
| 주황 (`COLOR_DESCENT`) | 직선 하강 미리보기 |
| 노랑 (`COLOR_START`) | 경로 시작점 (구체 강조) |
| 초록 (`COLOR_SUCCESS`) | 실행 성공 후 |
| 빨강 (`COLOR_FAIL`) | 실행 실패 후 |

각 경로마다 텍스트 라벨은 **결과 텍스트 한 줄** (`Square: OK` / `Descent: OK`) 만 둔다.
waypoint 마다 라벨을 찍지 않는다 — 두 경로의 시작점이 같은 좌표에 있어 겹쳐 보이기 때문.

또한 같은 ns 의 이전 마커는 새 publish 전에 RViz 에서 `DELETE` 한다.
이렇게 하면 셀을 여러 번 실행해도 마커가 누적되지 않는다.

In [ ]:
from visualization_msgs.msg import Marker
from std_msgs.msg import ColorRGBA

COLOR_SQUARE  = ColorRGBA(r=0.2, g=0.8, b=1.0, a=0.9)
COLOR_DESCENT = ColorRGBA(r=1.0, g=0.6, b=0.0, a=0.9)
COLOR_START   = ColorRGBA(r=1.0, g=1.0, b=0.0, a=0.9)
COLOR_SUCCESS = ColorRGBA(r=0.0, g=1.0, b=0.0, a=0.9)
COLOR_FAIL    = ColorRGBA(r=1.0, g=0.0, b=0.0, a=0.9)

_markers = MarkerArray()
_marker_id = {'next': 0}

def _next_id() -> int:
    _marker_id['next'] += 1
    return _marker_id['next']

def _delete_in_ns(ns_prefix: str) -> None:
    '''같은 ns 의 기존 마커를 RViz 에서 DELETE 하고 _markers 에서도 제거.
    셀 재실행 시 마커가 누적되어 텍스트가 겹치는 것을 막는다.'''
    targets = [m for m in _markers.markers if m.ns.startswith(ns_prefix)]
    if not targets:
        return
    delete_array = MarkerArray()
    stamp = node.get_clock().now().to_msg()
    for m in targets:
        d = Marker()
        d.header.frame_id = REFERENCE_FRAME
        d.header.stamp = stamp
        d.ns = m.ns
        d.id = m.id
        d.action = Marker.DELETE
        delete_array.markers.append(d)
    marker_pub.publish(delete_array)
    _markers.markers = [m for m in _markers.markers if not m.ns.startswith(ns_prefix)]

def publish_path_markers(waypoints, start_pose: Pose,
                         ns: str, color: ColorRGBA, label: str) -> None:
    '''경로 미리보기: line strip + waypoint 점 + 노란 시작 구체.
    텍스트 라벨은 publish_result_marker 가 결과 한 줄만 추가한다.'''
    _delete_in_ns(ns + '_')
    stamp = node.get_clock().now().to_msg()
    pts = [start_pose.position] + [wp.position for wp in waypoints]

    line = Marker()
    line.header.frame_id = REFERENCE_FRAME
    line.header.stamp = stamp
    line.ns = ns + '_path'
    line.id = _next_id()
    line.type = Marker.LINE_STRIP
    line.action = Marker.ADD
    line.pose.orientation.w = 1.0
    line.scale.x = 0.008
    line.color = color
    line.points = [Point(x=p.x, y=p.y, z=p.z) for p in pts]
    _markers.markers.append(line)

    spheres = Marker()
    spheres.header.frame_id = REFERENCE_FRAME
    spheres.header.stamp = stamp
    spheres.ns = ns + '_waypoints'
    spheres.id = _next_id()
    spheres.type = Marker.SPHERE_LIST
    spheres.action = Marker.ADD
    spheres.pose.orientation.w = 1.0
    spheres.scale = Vector3(x=0.025, y=0.025, z=0.025)
    spheres.color = color
    spheres.points = [Point(x=p.x, y=p.y, z=p.z) for p in pts]
    _markers.markers.append(spheres)

    start_sphere = Marker()
    start_sphere.header.frame_id = REFERENCE_FRAME
    start_sphere.header.stamp = stamp
    start_sphere.ns = ns + '_start'
    start_sphere.id = _next_id()
    start_sphere.type = Marker.SPHERE
    start_sphere.action = Marker.ADD
    start_sphere.pose.position = Point(x=start_pose.position.x,
                                       y=start_pose.position.y,
                                       z=start_pose.position.z)
    start_sphere.pose.orientation.w = 1.0
    start_sphere.scale = Vector3(x=0.04, y=0.04, z=0.04)
    start_sphere.color = COLOR_START
    _markers.markers.append(start_sphere)

    marker_pub.publish(_markers)

def publish_result_marker(ns: str, success: bool, label: str) -> None:
    '''LINE_STRIP 색을 결과 색으로 바꾸고 결과 텍스트 한 개만 추가.'''
    _delete_in_ns(ns + '_result')
    stamp = node.get_clock().now().to_msg()
    color = COLOR_SUCCESS if success else COLOR_FAIL
    center = None
    for m in _markers.markers:
        if m.ns == ns + '_path':
            m.color = color
            m.header.stamp = stamp
            if m.points:
                mid = m.points[len(m.points) // 2]
                center = Point(x=mid.x, y=mid.y, z=mid.z + 0.08)
    if center is not None:
        result = Marker()
        result.header.frame_id = REFERENCE_FRAME
        result.header.stamp = stamp
        result.ns = ns + '_result'
        result.id = _next_id()
        result.type = Marker.TEXT_VIEW_FACING
        result.action = Marker.ADD
        result.pose.position = center
        result.pose.orientation.w = 1.0
        result.scale.z = 0.06
        result.color = color
        result.text = f'{label}: {"OK" if success else "FAIL"}'
        _markers.markers.append(result)
    marker_pub.publish(_markers)

## 4. 실행 시나리오

여기부터는 위에서 정의한 핵심 함수와 헬퍼를 호출만 한다.

- `ready` 자세로 초기화
- 정사각형 시작점으로 이동 (Pose Goal)
- **정사각형 직선 경로** — `plan_and_run_cartesian()` 한 번
- **직선 하강** — `plan_and_run_cartesian()` 한 번
- `ready` 복귀

### 4-1. `ready` 자세로 초기화

In [ ]:
node.get_logger().info('--- ready 자세로 초기 이동 ---')
go_to_joint_goal(ready_target)
time.sleep(1.0)

### 4-2. 정사각형 시작점으로 이동 (Pose Goal)

정사각형은 끝단이 **아래를 향한 자세** (`roll=π`) 로 X-Y 평면 (`z=0.40`) 에 그린다.
한 변 **0.20 m** (이전 0.15 m 에서 확장). 시작점은 `(SX, -SIDE/2, 0.40)`.
FR3 reach ~855 mm 안에서 안전한 영역.

In [ ]:
SQUARE_Z = 0.40                # 정사각형 평면 z
SIDE     = 0.20                # 한 변 20 cm (이전 15 cm 에서 확장)
SX, SY   = 0.40, -SIDE / 2     # 시작점 (X-Y) — base 기준 ~40 cm 전방, 좌우 대칭

square_orientation = euler_to_quaternion(math.pi, 0.0, 0.0)  # 그리퍼 아래

start_pose = Pose()
start_pose.position = Point(x=SX, y=SY, z=SQUARE_Z)
start_pose.orientation = square_orientation

node.get_logger().info(
    f'--- 정사각형 시작점으로 이동: ({SX:.2f}, {SY:.3f}, {SQUARE_Z:.2f}) ---'
)
ok = go_to_pose_goal(start_pose)
if not ok:
    node.get_logger().error('시작점 이동 실패 — ready 로 복귀')
    go_to_joint_goal(ready_target)
time.sleep(1.0)

### 4-3. 정사각형 Cartesian 경로 — `plan_and_run_cartesian()` 한 번

X-Y 평면에서 한 변 0.20 m 정사각형을 시계 방향으로 그린다
(시작점 → +y → +x → -y → -x → 시작점 복귀).

**여기서 본 예제의 핵심 패턴이 한 줄로 압축된다.**
미리보기 마커 → 계획+게이트+실행 → 결과 마커.

In [ ]:
waypoints = [
    Pose(position=Point(x=SX,        y=SY + SIDE, z=SQUARE_Z), orientation=square_orientation),
    Pose(position=Point(x=SX + SIDE, y=SY + SIDE, z=SQUARE_Z), orientation=square_orientation),
    Pose(position=Point(x=SX + SIDE, y=SY,        z=SQUARE_Z), orientation=square_orientation),
    Pose(position=Point(x=SX,        y=SY,        z=SQUARE_Z), orientation=square_orientation),
]

publish_path_markers(waypoints, start_pose,
                     ns='square', color=COLOR_SQUARE, label='Square')

ok = plan_and_run_cartesian(waypoints, label='Square', max_step=0.02)
publish_result_marker('square', ok, 'Square')
time.sleep(1.0)

### 4-4. 직선 하강 시연

정사각형 시작점에서 **직선으로 0.20 m 아래** 로 내려보낸다 (이전 0.15 m 에서 확장).
`z` 값만 단조감소. 7 개 waypoint (≈ 2.86 cm 간격) 로 보간이 매끄럽다.

In [ ]:
DESCENT = 0.20  # 20 cm (이전 15 cm 에서 확장)
descent_start = Pose()
descent_start.position = Point(x=SX, y=SY, z=SQUARE_Z)
descent_start.orientation = square_orientation

descent_waypoints = []
for i in range(1, 8):
    p = Pose()
    p.position = Point(x=SX, y=SY, z=SQUARE_Z - i * (DESCENT / 7))
    p.orientation = square_orientation
    descent_waypoints.append(p)

publish_path_markers(descent_waypoints, descent_start,
                     ns='descent', color=COLOR_DESCENT, label='Descent')

ok = plan_and_run_cartesian(descent_waypoints, label='Descent', max_step=0.02)
publish_result_marker('descent', ok, 'Descent')
time.sleep(1.0)

### 4-5. `ready` 자세로 복귀

In [ ]:
node.get_logger().info('--- ready 자세로 복귀 ---')
go_to_joint_goal(ready_target)
node.get_logger().info('=== franka_ex05 완료! ===')

## 5. 정리

노트북을 닫기 전에 노드와 rclpy 를 안전하게 정리한다.

In [ ]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass